In [6]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
import joblib
import config as cfg

df = pd.read_csv('../data/training_gps.csv')
df = cfg.add_features(df)
X  = df[cfg.FEATURE_COLS].values
print("Data loaded:", df.shape)
print("Features:", cfg.FEATURE_COLS)
df.head()

Data loaded: (760, 8)
Features: ['lat_rad', 'lng_rad', 'hour_sin', 'hour_cos']


,lat,lng,hour,zone,lat_rad,lng_rad,hour_sin,hour_cos
0,18.113487,83.395421,8,known,0.316140,1.455525,8.660254e-01,-0.500000
1,18.110947,83.397728,11,known,0.316096,1.455565,2.588190e-01,-0.965926
2,18.114091,83.402212,8,known,0.316151,1.455643,8.660254e-01,-0.500000
3,18.117592,83.394227,12,known,0.316212,1.455504,1.224647e-16,-1.000000
4,18.110563,83.393066,11,known,0.316089,1.455484,2.588190e-01,-0.965926


In [7]:
db = DBSCAN(eps=cfg.DBSCAN_EPS,
            min_samples=cfg.DBSCAN_MIN_PTS,
            metric='haversine')
db.fit(X[:, :2])

df['cluster'] = db.labels_
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise    = list(db.labels_).count(-1)

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise}")
df['cluster'].value_counts()

Clusters found : 1
Noise points   : 0


cluster
0    760
Name: count, dtype: int64

In [8]:
iso = IsolationForest(contamination=cfg.ISO_CONTAMINATION,
                      random_state=42)
iso.fit(X)
df['iso_score'] = iso.decision_function(X)
df['iso_pred']  = iso.predict(X)

print("Anomalies by IsoForest:", (df['iso_pred'] == -1).sum())
print("Normal points         :", (df['iso_pred'] ==  1).sum())

Anomalies by IsoForest: 38
Normal points         : 722


In [9]:
df['anomaly'] = (df['cluster'] == -1) | (df['iso_pred'] == -1)

print("Total anomalies flagged:", df['anomaly'].sum())
print("Total safe points      :", (~df['anomaly']).sum())
df[df['anomaly']].head(10)

Total anomalies flagged: 38
Total safe points      : 722


,lat,lng,hour,zone,lat_rad,lng_rad,hour_sin,hour_cos,cluster,iso_score,iso_pred,anomaly
172,18.112280,83.419097,8,known,0.316119,1.455938,0.866025,-5.000000e-01,0,-0.012502,-1,True
283,18.116942,83.377521,8,known,0.316200,1.455212,0.866025,-5.000000e-01,0,-0.008114,-1,True
299,18.118415,83.380923,8,known,0.316226,1.455272,0.866025,-5.000000e-01,0,-0.001751,-1,True
480,18.146456,83.406744,19,known,0.316715,1.455722,-0.965926,2.588190e-01,0,-0.010532,-1,True
485,18.152919,83.404631,19,known,0.316828,1.455685,-0.965926,2.588190e-01,0,-0.030024,-1,True
543,18.152572,83.409247,18,known,0.316822,1.455766,-1.000000,-1.836970e-16,0,-0.029285,-1,True
552,18.139928,83.405112,8,known,0.316601,1.455694,0.866025,-5.000000e-01,0,-0.000575,-1,True
554,18.141900,83.411029,9,known,0.316636,1.455797,0.707107,-7.071068e-01,0,-0.008211,-1,True
560,18.142437,83.374475,18,known,0.316645,1.455159,-1.000000,-1.836970e-16,0,-0.011060,-1,True
564,18.151942,83.377520,8,known,0.316811,1.455212,0.866025,-5.000000e-01,0,-0.013023,-1,True


In [10]:
joblib.dump(db,  '../models/dbscan.pkl')
joblib.dump(iso, '../models/isoforest.pkl')
df.to_csv('../data/scored_points.csv', index=False)

print("dbscan.pkl     saved")
print("isoforest.pkl  saved")
print("scored_points.csv saved")
print("\nAll done! Ready for visualization")

dbscan.pkl     saved
isoforest.pkl  saved
scored_points.csv saved

All done! Ready for visualization
